# Lab — Building an Enterprise Visual Similarity and Retrieval System

One manufacturing archive contains component families, defect types, repeated captures across Factory A/B/C, near duplicates, source shortcuts, and controlled label noise. We will define three different meanings of relevance, evaluate frozen and metric-learned representations, mine hard negatives, compare whole-image and region retrieval, verify exact search, sweep an approximate FAISS index, and save a versioned evidence artifact.

This lab is CPU-safe and credential-free. The official torchvision ResNet-18 weights are a public download and use the maintained weight/preprocessing API. The DINOv2 extension is opt-in and disabled by default. No hosted vector database, biometric identity data, external write, or physical action is used.


## 0. Experiment and safety contract

**Success criteria:** metric math and ranking metrics pass assertion-backed examples; Factory C never supplies training gradients; every ranking names its relevance contract; ANN is compared with an exact baseline; local, optional, and assumed evidence remain separate.

**Risk boundaries:** images are procedural, identity IDs describe manufactured components rather than people, region boxes/masks are oracle inputs, and runtime thresholds are demonstrations only. The notebook does not certify production quality, privacy, latency, capacity, or legal suitability.

![A query region is encoded, searched against a versioned index, filtered by policy, and returned as ranked evidence.](assets/retrieval-pipeline.svg)


In [ ]:
from __future__ import annotations

import copy
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import statistics
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageDraw, ImageEnhance
import sklearn
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

SEED = 37
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
DEVICE = torch.device("cpu")

cwd = Path.cwd().resolve()
relative_course = Path("curriculum/beginner/07-visual-embeddings-metric-learning-retrieval")
COURSE_DIR = cwd / relative_course if (cwd / relative_course).exists() else cwd
ARTIFACT_DIR = COURSE_DIR / ".artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FULL_RUN = os.getenv("CV_FULL_RUN", "0") == "1"
RUN = {
    "image_size": 96,
    "identities_per_family_defect": 5 if FULL_RUN else 4,
    "metric_epochs": 10 if FULL_RUN else 6,
    "steps_per_epoch": 18 if FULL_RUN else 10,
    "ann_vectors": 20_000 if FULL_RUN else 8_000,
    "ann_queries": 200 if FULL_RUN else 100,
}
DEMONSTRATION_THRESHOLD_NOTICE = "Demonstration thresholds for this notebook runtime only."
DEMONSTRATION_CONTRACT = {
    "notice": DEMONSTRATION_THRESHOLD_NOTICE,
    "ann_recall_at_10_min": 0.95,
    "p95_individual_query_ms_max": 10.0,
    "same_source_rate_at_5_max": 0.45,
}

VERSIONS = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": PIL.__version__,
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "faiss": importlib.metadata.version("faiss-cpu"),
    "device": str(DEVICE),
    "seed": SEED,
}
print(json.dumps({"run": RUN, "versions": VERSIONS, "threshold_notice": DEMONSTRATION_THRESHOLD_NOTICE}, indent=2))


## 1. Generate a source-aware retrieval corpus

The same component identity is rendered under three capture styles. The object mask and box support oracle region experiments. A subset receives a near-duplicate copy; a small controlled subset has an observed defect label that differs from the hidden true label so hard-negative review can expose annotation problems.

Factory C is held out from metric-learning updates. Duplicate groups stay explicit so they can be excluded from ordinary retrieval splits.


In [ ]:
FAMILIES = ["gear", "bearing", "valve", "bracket"]
DEFECTS = ["normal", "scratch", "corrosion"]
SOURCES = ["A", "B", "C"]
SOURCE_COLORS = {"A": (42, 66, 115), "B": (38, 105, 72), "C": (120, 61, 54)}
FAMILY_COLORS = {
    "gear": (220, 190, 75),
    "bearing": (165, 185, 205),
    "valve": (115, 180, 215),
    "bracket": (205, 145, 95),
}


def stable_seed(*parts: object) -> int:
    token = "|".join(map(str, parts)).encode()
    return int(hashlib.sha256(token).hexdigest()[:8], 16)


def shape_mask(family: str, bbox: tuple[int, int, int, int], identity_index: int) -> Image.Image:
    mask = Image.new("L", (RUN["image_size"], RUN["image_size"]), 0)
    draw = ImageDraw.Draw(mask)
    x0, y0, x1, y1 = bbox
    if family == "gear":
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        radius = min(x1 - x0, y1 - y0) / 2
        points = []
        for i in range(24):
            angle = 2 * math.pi * i / 24
            r = radius if i % 2 == 0 else radius * 0.82
            points.append((cx + r * math.cos(angle), cy + r * math.sin(angle)))
        draw.polygon(points, fill=255)
        draw.ellipse((cx - radius * 0.26, cy - radius * 0.26, cx + radius * 0.26, cy + radius * 0.26), fill=0)
    elif family == "bearing":
        draw.ellipse(bbox, fill=255)
        inset = 13 + identity_index % 3
        draw.ellipse((x0 + inset, y0 + inset, x1 - inset, y1 - inset), fill=0)
    elif family == "valve":
        cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
        draw.rounded_rectangle((x0 + 8, y0 + 12, x1 - 8, y1 - 5), radius=10, fill=255)
        draw.rectangle((cx - 5, y0, cx + 5, y0 + 18), fill=255)
        draw.ellipse((cx - 15, y0 - 4, cx + 15, y0 + 10), fill=255)
    else:
        thickness = 16 + identity_index % 4
        draw.rectangle((x0, y0, x0 + thickness, y1), fill=255)
        draw.rectangle((x0, y1 - thickness, x1, y1), fill=255)
        draw.ellipse((x0 + 4, y0 + 7, x0 + thickness - 4, y0 + 19), fill=0)
    return mask


def render_sample(family: str, defect: str, source: str, identity_index: int) -> tuple[Image.Image, Image.Image, tuple[int, int, int, int]]:
    rng = np.random.default_rng(stable_seed(family, defect, source, identity_index, SEED))
    size = RUN["image_size"]
    background = np.zeros((size, size, 3), dtype=np.int16)
    base = np.array(SOURCE_COLORS[source], dtype=np.int16)
    background[:] = base
    gradient = np.linspace(-18, 18, size, dtype=np.int16)
    if source == "A":
        background += gradient[:, None, None]
    elif source == "B":
        background += gradient[None, :, None]
    else:
        background += ((np.add.outer(np.arange(size), np.arange(size)) % 16) < 3)[..., None] * 24
    background += rng.normal(0, 5, background.shape).astype(np.int16)
    image = Image.fromarray(np.clip(background, 0, 255).astype(np.uint8), "RGB")
    draw = ImageDraw.Draw(image)
    if source == "A":
        for y in range(8, size, 12): draw.line((0, y, size, y), fill=(67, 91, 142), width=2)
    elif source == "B":
        for x in range(7, size, 13): draw.line((x, 0, x, size), fill=(60, 132, 94), width=2)
    else:
        for offset in range(-size, size, 17): draw.line((offset, 0, offset + size, size), fill=(146, 76, 67), width=2)

    jitter_x, jitter_y = int(rng.integers(-4, 5)), int(rng.integers(-4, 5))
    bbox = (22 + jitter_x, 20 + jitter_y, 75 + jitter_x, 75 + jitter_y)
    mask = shape_mask(family, bbox, identity_index)
    object_color = np.array(FAMILY_COLORS[family], dtype=np.int16)
    source_tint = {"A": np.array((8, 0, 12)), "B": np.array((-8, 12, 0)), "C": np.array((15, -5, -8))}[source]
    object_color = tuple(np.clip(object_color + source_tint + int(rng.integers(-12, 13)), 20, 240).astype(int))
    object_layer = Image.new("RGB", image.size, object_color)
    image.paste(object_layer, mask=mask)
    draw = ImageDraw.Draw(image)
    x0, y0, x1, y1 = bbox
    mark_x = x0 + 12 + (identity_index * 7) % max(16, x1 - x0 - 22)
    draw.ellipse((mark_x, y1 - 14, mark_x + 5, y1 - 9), fill=(40, 40, 40))
    if defect == "scratch":
        draw.line((x0 + 12, y0 + 18 + identity_index % 8, x1 - 11, y1 - 18), fill=(245, 245, 235), width=3)
        draw.line((x0 + 12, y0 + 20 + identity_index % 8, x1 - 11, y1 - 16), fill=(70, 60, 55), width=1)
    elif defect == "corrosion":
        for j in range(5):
            cx = int(rng.integers(x0 + 10, x1 - 9)); cy = int(rng.integers(y0 + 10, y1 - 9))
            if mask.getpixel((cx, cy)):
                radius = 2 + (j + identity_index) % 3
                draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=(150, 74, 32))
    return image, mask, bbox


samples: list[dict] = []
for family in FAMILIES:
    for defect in DEFECTS:
        for identity_index in range(RUN["identities_per_family_defect"]):
            identity_id = f"{family}-{defect}-{identity_index:02d}"
            for source in SOURCES:
                image, mask, bbox = render_sample(family, defect, source, identity_index)
                sample_id = f"{identity_id}-{source}"
                observed = defect
                if source == "B" and identity_index == 0 and family in {"gear", "bracket"}:
                    observed = DEFECTS[(DEFECTS.index(defect) + 1) % len(DEFECTS)]
                samples.append({
                    "sample_id": sample_id,
                    "image": image,
                    "mask": mask,
                    "bbox": bbox,
                    "family": family,
                    "true_defect": defect,
                    "observed_defect": observed,
                    "source": source,
                    "identity_id": identity_id,
                    "duplicate_group": sample_id,
                    "is_near_duplicate": False,
                    "label_error": observed != defect,
                })

# Near duplicates are same-capture variants, not same-identity captures from another source.
duplicate_bases = [s for s in samples if s["source"] == "B" and s["sample_id"].endswith(("00-B", "01-B"))]
for base in duplicate_bases:
    enhanced = ImageEnhance.Brightness(base["image"]).enhance(1.035)
    shifted = Image.new("RGB", enhanced.size, SOURCE_COLORS[base["source"]])
    shifted.paste(enhanced.crop((1, 0, enhanced.width, enhanced.height)), (0, 0))
    dup = dict(base)
    dup.update({
        "sample_id": base["sample_id"] + "-near-dup",
        "image": shifted,
        "mask": base["mask"].copy(),
        "duplicate_group": base["sample_id"],
        "is_near_duplicate": True,
    })
    samples.append(dup)

metadata = pd.DataFrame([{k: v for k, v in s.items() if k not in {"image", "mask"}} for s in samples])
assert metadata.sample_id.is_unique
assert not metadata.query("source == 'C'").label_error.any()
print(metadata.groupby(["source", "is_near_duplicate"]).size().unstack(fill_value=0))
print("samples:", len(samples), "identities:", metadata.identity_id.nunique(), "duplicate groups:", metadata.duplicate_group.nunique())
display(metadata.head())


In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for axis, sample in zip(axes.flat, [s for s in samples if not s["is_near_duplicate"]][:18]):
    axis.imshow(sample["image"])
    axis.set_title(f"{sample['family']} · {sample['true_defect']} · {sample['source']}", fontsize=8)
    axis.axis("off")
plt.tight_layout()
plt.show()

split_contract = pd.DataFrame([
    {"role": "metric train", "sources": "A/B", "duplicates": "excluded", "labels": "observed defect"},
    {"role": "held-out queries", "sources": "C", "duplicates": "excluded", "labels": "true relevance only"},
    {"role": "exact gallery", "sources": "A/B", "duplicates": "excluded", "labels": "true relevance only"},
    {"role": "duplicate evaluation", "sources": "B", "duplicates": "explicit grouped pairs", "labels": "same capture group"},
])
display(split_contract)


## 2. Make three similarity contracts executable

Component retrieval treats the same family as relevant. Defect retrieval treats the same hidden defect as relevant. Identity retrieval asks for the exact manufactured identity across sources. Duplicate detection later uses same-capture duplicate groups.

The relevance function—not the vector database—defines success.


In [ ]:
def l2_normalize(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array, dtype=np.float32)
    return array / np.clip(np.linalg.norm(array, axis=1, keepdims=True), 1e-12, None)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return l2_normalize(a) @ l2_normalize(b).T


def relevance_key(sample: dict, contract: str) -> str:
    if contract == "component": return sample["family"]
    if contract == "defect": return sample["true_defect"]
    if contract == "identity": return sample["identity_id"]
    if contract == "duplicate": return sample["duplicate_group"]
    raise KeyError(contract)


def precision_at_k(relevance: np.ndarray, k: int) -> float:
    return float(np.asarray(relevance[:k], dtype=float).sum() / k)


def recall_at_k(relevance: np.ndarray, k: int, relevant_total: int) -> float:
    return float(np.asarray(relevance[:k], dtype=float).sum() / relevant_total) if relevant_total else float("nan")


def average_precision(relevance: np.ndarray, relevant_total: int) -> float:
    if relevant_total == 0: return float("nan")
    hits = 0
    score = 0.0
    for rank, relevant in enumerate(np.asarray(relevance, dtype=bool), start=1):
        if relevant:
            hits += 1
            score += hits / rank
    return score / relevant_total


def reciprocal_rank(relevance: np.ndarray) -> float:
    positions = np.flatnonzero(relevance)
    return float(1 / (positions[0] + 1)) if len(positions) else 0.0


known = np.array([1, 0, 1, 0], dtype=bool)
assert precision_at_k(known, 2) == 0.5
assert recall_at_k(known, 2, 2) == 0.5
assert np.isclose(average_precision(known, 2), (1 + 2 / 3) / 2)
assert reciprocal_rank(known) == 1.0
assert math.isnan(average_precision(np.zeros(3, dtype=bool), 0))

rng = np.random.default_rng(SEED)
a = l2_normalize(rng.normal(size=(6, 9)).astype(np.float32))
b = l2_normalize(rng.normal(size=(6, 9)).astype(np.float32))
cos = np.sum(a * b, axis=1)
euclidean_squared = np.sum((a - b) ** 2, axis=1)
normalization_identity_error = float(np.max(np.abs(euclidean_squared - (2 - 2 * cos))))
assert normalization_identity_error < 1e-5
print("max normalized L2/cosine identity error:", normalization_identity_error)


## 3. Verify contrastive and triplet objectives

The functions below expose the distances, hinge, and active-loss fraction. `torch.nn` packages these ideas, but the primitive must be visible before mining policies are compared.

![Two inputs pass through shared-weight encoders before their embeddings are compared by a metric-learning objective.](assets/siamese-network.svg)


In [ ]:
def contrastive_pair_loss(z1: torch.Tensor, z2: torch.Tensor, similar: torch.Tensor, margin: float = 1.0) -> torch.Tensor:
    distance = F.pairwise_distance(z1, z2)
    return (similar * distance.square() + (1 - similar) * F.relu(margin - distance).square()).mean()


def triplet_loss_with_diagnostics(anchor: torch.Tensor, positive: torch.Tensor, negative: torch.Tensor, margin: float = 0.35):
    positive_distance = F.pairwise_distance(anchor, positive)
    negative_distance = F.pairwise_distance(anchor, negative)
    hinge = positive_distance - negative_distance + margin
    return F.relu(hinge).mean(), float((hinge > 0).float().mean().detach())


anchor = torch.tensor([[1.0, 0.0], [1.0, 0.0]])
positive = torch.tensor([[0.9, 0.1], [0.0, 1.0]])
negative = torch.tensor([[0.0, 1.0], [0.8, 0.2]])
easy_loss, _ = triplet_loss_with_diagnostics(anchor[:1], positive[:1], negative[:1], margin=0.2)
hard_loss, _ = triplet_loss_with_diagnostics(anchor[1:], positive[1:], negative[1:], margin=0.2)
assert easy_loss.item() == 0.0 and hard_loss.item() > 0
same_pair = contrastive_pair_loss(anchor[:1], anchor[:1], torch.ones(1))
far_negative = contrastive_pair_loss(anchor[:1], negative[:1], torch.zeros(1), margin=0.5)
assert same_pair.item() < 1e-10 and far_negative.item() == 0.0
display(pd.DataFrame({"case": ["easy triplet", "hard triplet"], "loss": [easy_loss.item(), hard_loss.item()]}))


## 4. Official supervised baseline and oracle region policies

We load official `ResNet18_Weights.DEFAULT`, preserve its preprocessing contract, remove the classifier, freeze the encoder, and extract normalized 512-dimensional features.

Three inputs answer different questions:

- full image: realistic context plus possible source shortcut;
- oracle object crop: target location supplied by ground truth;
- oracle masked region: target location and exact mask supplied by ground truth.

Crop/mask scores are ceilings for region preprocessing—not end-to-end detector/segmenter evidence.


In [ ]:
def region_image(sample: dict, mode: str) -> Image.Image:
    if mode == "full": return sample["image"]
    if mode == "crop": return sample["image"].crop(sample["bbox"])
    if mode == "masked":
        neutral = Image.new("RGB", sample["image"].size, (112, 112, 112))
        return Image.composite(sample["image"], neutral, sample["mask"])
    raise KeyError(mode)


weights = ResNet18_Weights.DEFAULT
resnet_encoder = resnet18(weights=weights)
resnet_dimension = resnet_encoder.fc.in_features
resnet_encoder.fc = nn.Identity()
for parameter in resnet_encoder.parameters(): parameter.requires_grad = False
resnet_encoder.eval().to(DEVICE)
resnet_transform = weights.transforms(crop_size=128, resize_size=144)


class RegionDataset(Dataset):
    def __init__(self, rows: list[dict], mode: str, transform):
        self.rows, self.mode, self.transform = rows, mode, transform
    def __len__(self): return len(self.rows)
    def __getitem__(self, index): return self.transform(region_image(self.rows[index], self.mode))


@torch.inference_mode()
def extract_resnet_features(rows: list[dict], mode: str) -> np.ndarray:
    loader = DataLoader(RegionDataset(rows, mode, resnet_transform), batch_size=16, shuffle=False, num_workers=0)
    values = [resnet_encoder(batch.to(DEVICE)).cpu().numpy() for batch in loader]
    return l2_normalize(np.concatenate(values).astype(np.float32))


resnet_features = {}
for mode in ["full", "crop", "masked"]:
    started = time.perf_counter()
    resnet_features[mode] = extract_resnet_features(samples, mode)
    print(mode, resnet_features[mode].shape, f"{time.perf_counter() - started:.2f}s")
assert resnet_features["full"].shape == (len(samples), resnet_dimension)


In [ ]:
original_indices = np.array([i for i, sample in enumerate(samples) if not sample["is_near_duplicate"]])
query_indices = np.array([i for i in original_indices if samples[i]["source"] == "C"])
gallery_indices = np.array([i for i in original_indices if samples[i]["source"] in {"A", "B"}])


def evaluate_retrieval(features: np.ndarray, queries: np.ndarray, gallery: np.ndarray, contract: str, k: int = 5) -> dict:
    scores = features[queries] @ features[gallery].T
    order = np.argsort(-scores, axis=1)
    rows = []
    for row_index, query_index in enumerate(queries):
        ranked = gallery[order[row_index]]
        query_key = relevance_key(samples[query_index], contract)
        gallery_keys = np.array([relevance_key(samples[i], contract) for i in gallery])
        ranked_relevance = np.array([relevance_key(samples[i], contract) == query_key for i in ranked])
        relevant_total = int((gallery_keys == query_key).sum())
        rows.append({
            "P@1": precision_at_k(ranked_relevance, 1),
            "P@5": precision_at_k(ranked_relevance, k),
            "R@5": recall_at_k(ranked_relevance, k, relevant_total),
            "AP": average_precision(ranked_relevance, relevant_total),
            "RR": reciprocal_rank(ranked_relevance),
            "relevant_total": relevant_total,
        })
    frame = pd.DataFrame(rows)
    eligible = frame.dropna(subset=["AP"])
    return {
        "P@1": float(frame["P@1"].mean()),
        "P@5": float(frame["P@5"].mean()),
        "R@5": float(frame["R@5"].mean()),
        "mAP": float(eligible["AP"].mean()),
        "MRR": float(frame["RR"].mean()),
        "eligible_queries": int(len(eligible)),
    }


baseline_rows = []
for mode, features in resnet_features.items():
    for contract in ["component", "defect", "identity"]:
        baseline_rows.append({"representation": "ImageNet ResNet-18", "region": mode, "contract": contract,
                              **evaluate_retrieval(features, query_indices, gallery_indices, contract)})
baseline_retrieval = pd.DataFrame(baseline_rows)
display(baseline_retrieval.round(3))
print("These are Factory C queries against an A/B gallery; oracle crop/mask rows include target-location information.")


## 5. Build a tiny metric encoder and `P × K` batches

The domain model optimizes **defect similarity**. It is intentionally small and educational. Every batch contains all three defect classes and multiple examples per class. We compare random, semi-hard, and batch-hard online mining while holding architecture, initialization, data, optimizer, margin, and update budget fixed.

![Triplet learning asks an anchor to be closer to a positive than to a negative by a declared margin.](assets/triplet-learning.svg)


In [ ]:
metric_transform = transforms.Compose([
    transforms.Resize((64, 64), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
DEFECT_TO_ID = {name: i for i, name in enumerate(DEFECTS)}
train_indices = np.array([i for i in original_indices if samples[i]["source"] in {"A", "B"}])


class TinyMetricEncoder(nn.Module):
    def __init__(self, dimension: int = 32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.projection = nn.Linear(64, dimension)
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        hidden = self.features(images).flatten(1)
        return F.normalize(self.projection(hidden), dim=1)


def image_tensor(index: int, mode: str = "masked") -> torch.Tensor:
    return metric_transform(region_image(samples[index], mode))


def make_pk_batch(rng: np.random.Generator, p: int = 3, k: int = 4):
    chosen_labels = rng.choice(len(DEFECTS), size=p, replace=False)
    batch_indices = []
    for label in chosen_labels:
        candidates = [i for i in train_indices if DEFECT_TO_ID[samples[i]["observed_defect"]] == label]
        batch_indices.extend(rng.choice(candidates, size=k, replace=False).tolist())
    images = torch.stack([image_tensor(i) for i in batch_indices]).to(DEVICE)
    labels = torch.tensor([DEFECT_TO_ID[samples[i]["observed_defect"]] for i in batch_indices], device=DEVICE)
    return images, labels, np.array(batch_indices)


def mine_online_triplets(embeddings: torch.Tensor, labels: torch.Tensor, policy: str, rng: np.random.Generator, margin: float):
    distances = torch.cdist(embeddings, embeddings, p=2)
    anchors, positives, negatives = [], [], []
    for anchor_index in range(len(labels)):
        positive_candidates = torch.where((labels == labels[anchor_index]) & (torch.arange(len(labels), device=labels.device) != anchor_index))[0]
        negative_candidates = torch.where(labels != labels[anchor_index])[0]
        if policy == "random":
            positive_index = int(rng.choice(positive_candidates.cpu().numpy()))
            negative_index = int(rng.choice(negative_candidates.cpu().numpy()))
        elif policy == "batch-hard":
            positive_index = int(positive_candidates[distances[anchor_index, positive_candidates].argmax()])
            negative_index = int(negative_candidates[distances[anchor_index, negative_candidates].argmin()])
        elif policy == "semi-hard":
            positive_index = int(rng.choice(positive_candidates.cpu().numpy()))
            positive_distance = distances[anchor_index, positive_index]
            candidate_distances = distances[anchor_index, negative_candidates]
            valid = negative_candidates[(candidate_distances > positive_distance) & (candidate_distances < positive_distance + margin)]
            negative_index = int(valid[distances[anchor_index, valid].argmin()]) if len(valid) else int(negative_candidates[candidate_distances.argmin()])
        else:
            raise KeyError(policy)
        anchors.append(anchor_index); positives.append(positive_index); negatives.append(negative_index)
    return torch.tensor(anchors), torch.tensor(positives), torch.tensor(negatives)


def train_metric_model(policy: str, seed: int = SEED):
    torch.manual_seed(seed)
    model = TinyMetricEncoder().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    rng = np.random.default_rng(seed)
    history = []
    margin = 0.35
    for epoch in range(RUN["metric_epochs"]):
        model.train()
        for step in range(RUN["steps_per_epoch"]):
            images, labels, _ = make_pk_batch(rng)
            embeddings = model(images)
            a, p, n = mine_online_triplets(embeddings, labels, policy, rng, margin)
            loss, active = triplet_loss_with_diagnostics(embeddings[a], embeddings[p], embeddings[n], margin)
            optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
            history.append({"policy": policy, "epoch": epoch + 1, "step": step + 1, "loss": float(loss.detach()), "active_fraction": active})
    return model.eval(), pd.DataFrame(history)


torch.manual_seed(SEED)
random_metric_model = TinyMetricEncoder().eval()
models, histories = {}, []
for policy in ["random", "semi-hard", "batch-hard"]:
    model, history = train_metric_model(policy)
    models[policy] = model
    histories.append(history)
training_history = pd.concat(histories, ignore_index=True)
display(training_history.groupby("policy").agg(final_loss=("loss", "last"), mean_active_fraction=("active_fraction", "mean")).round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for policy, frame in training_history.groupby("policy"):
    by_epoch = frame.groupby("epoch").mean(numeric_only=True)
    axes[0].plot(by_epoch.index, by_epoch.loss, marker="o", label=policy)
    axes[1].plot(by_epoch.index, by_epoch.active_fraction, marker="o", label=policy)
axes[0].set(title="Triplet loss", xlabel="epoch", ylabel="mean loss")
axes[1].set(title="Non-zero triplet fraction", xlabel="epoch", ylabel="active fraction")
for axis in axes: axis.grid(alpha=.25); axis.legend()
plt.tight_layout(); plt.show()


@torch.inference_mode()
def extract_metric_features(model: nn.Module, rows: list[dict], mode: str = "masked") -> np.ndarray:
    model.eval()
    tensors = [metric_transform(region_image(sample, mode)) for sample in rows]
    features = []
    for start in range(0, len(tensors), 32):
        features.append(model(torch.stack(tensors[start:start + 32]).to(DEVICE)).cpu().numpy())
    return l2_normalize(np.concatenate(features).astype(np.float32))


metric_features = {"random-init": extract_metric_features(random_metric_model, samples)}
for policy, model in models.items(): metric_features[policy] = extract_metric_features(model, samples)


def sampling_signal_table(features: np.ndarray, indices: np.ndarray, margin: float = 0.35) -> pd.DataFrame:
    # Compare the learning signal produced by easy, random, and hard negatives.
    rng = np.random.default_rng(SEED)
    vectors = features[indices]
    labels = np.array([samples[int(i)]["observed_defect"] for i in indices])
    distances = np.sqrt(np.maximum(0.0, 2.0 - 2.0 * (vectors @ vectors.T)))
    records = []
    for local_anchor in range(len(indices)):
        positives = np.where((labels == labels[local_anchor]) & (np.arange(len(indices)) != local_anchor))[0]
        negatives = np.where(labels != labels[local_anchor])[0]
        positive = int(rng.choice(positives))
        for policy, negative in {
            "easy (farthest)": int(negatives[np.argmax(distances[local_anchor, negatives])]),
            "random": int(rng.choice(negatives)),
            "hard (nearest)": int(negatives[np.argmin(distances[local_anchor, negatives])]),
        }.items():
            d_ap = float(distances[local_anchor, positive])
            d_an = float(distances[local_anchor, negative])
            records.append({"policy": policy, "positive_distance": d_ap, "negative_distance": d_an,
                            "hinge_value": max(0.0, d_ap - d_an + margin)})
    frame = pd.DataFrame(records)
    return frame.groupby("policy", sort=False).agg(
        mean_positive_distance=("positive_distance", "mean"),
        mean_negative_distance=("negative_distance", "mean"),
        active_triplet_fraction=("hinge_value", lambda values: float(np.mean(np.asarray(values) > 0))),
        mean_hinge_signal=("hinge_value", "mean"),
    ).reset_index()


sampler_signal = sampling_signal_table(metric_features["semi-hard"], train_indices)
display(sampler_signal.round(3))
print("Easy negatives can satisfy the margin and contribute zero gradient; hard negatives carry more signal but need review.")

comparison_rows = []
for name, features in metric_features.items():
    for contract in ["defect", "component", "identity"]:
        comparison_rows.append({"representation": name, "region": "masked", "contract": contract,
                                **evaluate_retrieval(features, query_indices, gallery_indices, contract)})
for mode in ["full", "crop", "masked"]:
    comparison_rows.append({"representation": "ImageNet ResNet-18", "region": mode, "contract": "defect",
                            **evaluate_retrieval(resnet_features[mode], query_indices, gallery_indices, "defect")})
representation_comparison = pd.DataFrame(comparison_rows)
display(representation_comparison.round(3))
print("The metric model was optimized for defect similarity; other contracts test what that objective discarded.")


## 6. Offline hard-negative feedback and simulated adjudication

Online mining is limited to the batch. Offline mining embeds the full training corpus and retrieves the nearest item with a different observed label. That item may be a useful hard negative—or evidence that the observed label is wrong. In this synthetic lab, hidden `true_defect` acts as an **oracle reviewer** and automatically adjudicates candidates. In production, hidden truth does not exist: a qualified human or domain workflow must review relevance and record the decision before a candidate becomes an approved negative.

![The hard-negative loop embeds the corpus, retrieves confusing wrong-label neighbours, reviews false negatives, and feeds approved examples back into training.](assets/hard-negative-mining.svg)


In [ ]:
def mine_offline_triplets(model: nn.Module) -> pd.DataFrame:
    train_samples = [samples[i] for i in train_indices]
    features = extract_metric_features(model, train_samples)
    similarity = features @ features.T
    observed = np.array([sample["observed_defect"] for sample in train_samples])
    true = np.array([sample["true_defect"] for sample in train_samples])
    rows = []
    rng = np.random.default_rng(SEED)
    for local_anchor, global_anchor in enumerate(train_indices):
        positives = np.where((observed == observed[local_anchor]) & (np.arange(len(observed)) != local_anchor))[0]
        negatives = np.where(observed != observed[local_anchor])[0]
        local_positive = int(rng.choice(positives))
        local_negative = int(negatives[np.argmax(similarity[local_anchor, negatives])])
        rows.append({
            "anchor_global": int(global_anchor),
            "positive_global": int(train_indices[local_positive]),
            "negative_global": int(train_indices[local_negative]),
            "similarity": float(similarity[local_anchor, local_negative]),
            "false_negative": bool(true[local_anchor] == true[local_negative]),
        })
    return pd.DataFrame(rows)


offline_model = copy.deepcopy(models["random"])
before_offline = evaluate_retrieval(extract_metric_features(offline_model, samples), query_indices, gallery_indices, "defect")
offline_triplets = mine_offline_triplets(offline_model)
offline_triplets["simulated_reviewer_decision"] = np.where(offline_triplets.false_negative, "reject: oracle says relevant", "approve as negative")
adjudication_contract = pd.DataFrame([
    {"environment": "synthetic lab", "adjudicator": "hidden true_defect oracle", "result": "simulated reviewer decision"},
    {"environment": "production", "adjudicator": "qualified human/domain workflow", "result": "auditable approved/rejected negative"},
])
display(adjudication_contract)
approved = offline_triplets.query("simulated_reviewer_decision == 'approve as negative'").reset_index(drop=True)
optimizer = torch.optim.AdamW(offline_model.parameters(), lr=8e-4, weight_decay=1e-4)
offline_history = []
offline_model.train()
for epoch in range(3):
    for start in range(0, len(approved), 24):
        batch = approved.iloc[start:start + 24]
        anchors = torch.stack([image_tensor(int(i)) for i in batch.anchor_global]).to(DEVICE)
        positives = torch.stack([image_tensor(int(i)) for i in batch.positive_global]).to(DEVICE)
        negatives = torch.stack([image_tensor(int(i)) for i in batch.negative_global]).to(DEVICE)
        za, zp, zn = offline_model(anchors), offline_model(positives), offline_model(negatives)
        loss, active = triplet_loss_with_diagnostics(za, zp, zn, 0.35)
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        offline_history.append({"epoch": epoch + 1, "loss": float(loss.detach()), "active_fraction": active})
offline_model.eval()
offline_features = extract_metric_features(offline_model, samples)
after_offline = evaluate_retrieval(offline_features, query_indices, gallery_indices, "defect")
offline_comparison = pd.DataFrame([
    {"stage": "before offline mining", **before_offline},
    {"stage": "after oracle-adjudicated offline mining", **after_offline},
])
display(offline_comparison.round(3))
print({"adjudication_mode": "simulated hidden-label oracle", "mined": len(offline_triplets), "approved": len(approved), "false_negative_rate": float(offline_triplets.false_negative.mean())})


## 7. Region retrieval and source shortcuts

For mixed-source queries, calculate how often neighbours share the true defect and how often they share the camera/source. A useful source-invariant defect embedding should increase semantic neighbour rate without simply clustering Factory A/B/C backgrounds.


In [ ]:
def neighbor_rates(features: np.ndarray, indices: np.ndarray, k: int = 5) -> dict:
    scores = features[indices] @ features[indices].T
    np.fill_diagonal(scores, -np.inf)
    ranked = np.argsort(-scores, axis=1)[:, :k]
    semantic, source, family = [], [], []
    for row, local_neighbors in enumerate(ranked):
        query = samples[int(indices[row])]
        neighbors = [samples[int(indices[j])] for j in local_neighbors]
        semantic.extend([item["true_defect"] == query["true_defect"] for item in neighbors])
        source.extend([item["source"] == query["source"] for item in neighbors])
        family.extend([item["family"] == query["family"] for item in neighbors])
    return {"semantic_neighbor_rate@5": float(np.mean(semantic)), "same_source_rate@5": float(np.mean(source)),
            "same_family_rate@5": float(np.mean(family))}


source_bias_rows = []
for mode, features in resnet_features.items():
    source_bias_rows.append({"representation": "ImageNet ResNet-18", "region": mode, **neighbor_rates(features, original_indices)})
for name, features in {**metric_features, "offline-oracle-adjudicated": offline_features}.items():
    source_bias_rows.append({"representation": name, "region": "masked", **neighbor_rates(features, original_indices)})
source_bias = pd.DataFrame(source_bias_rows)
source_bias["semantic_minus_source"] = source_bias["semantic_neighbor_rate@5"] - source_bias["same_source_rate@5"]
display(source_bias.round(3).sort_values("semantic_minus_source", ascending=False))

fig, axis = plt.subplots(figsize=(6, 4))
axis.scatter(source_bias["same_source_rate@5"], source_bias["semantic_neighbor_rate@5"])
for _, row in source_bias.iterrows():
    axis.annotate(f"{row.representation}\n{row.region}", (row["same_source_rate@5"], row["semantic_neighbor_rate@5"]), fontsize=7)
axis.set(xlabel="same-source neighbour rate@5", ylabel="semantic neighbour rate@5", title="Semantic structure versus source shortcut")
axis.grid(alpha=.25); plt.tight_layout(); plt.show()


## 8. Duplicate detection is a separate task

Semantic encoders may map two different examples close because they share a concept; perceptual hashes may be better for re-encoded or slightly shifted copies. We assign every underlying `duplicate_group` to development or test **before** constructing pairs, build positives and negatives only from groups in the same partition, select each threshold on development pairs, then report held-out precision and recall.


In [ ]:
def difference_hash(image: Image.Image, hash_size: int = 8) -> np.ndarray:
    grayscale = image.convert("L").resize((hash_size + 1, hash_size))
    pixels = np.asarray(grayscale)
    return (pixels[:, 1:] > pixels[:, :-1]).reshape(-1)


def hash_similarity(first: Image.Image, second: Image.Image) -> float:
    return float(1 - np.mean(difference_hash(first) != difference_hash(second)))


sample_lookup = {sample["sample_id"]: index for index, sample in enumerate(samples)}
all_duplicate_groups = sorted({sample["duplicate_group"] for sample in samples})
duplicate_group_split = {
    group: ("development" if stable_seed("duplicate-split", group) % 5 in {0, 1} else "test")
    for group in all_duplicate_groups
}
pair_rows = []
for duplicate in [sample for sample in samples if sample["is_near_duplicate"]]:
    anchor_group = duplicate["duplicate_group"]
    pair_split = duplicate_group_split[anchor_group]
    original_index = sample_lookup[anchor_group]
    duplicate_index = sample_lookup[duplicate["sample_id"]]
    pair_rows.append({"first": original_index, "second": duplicate_index, "duplicate": 1, "pair_type": "near duplicate",
                      "anchor_group": anchor_group, "candidate_group": anchor_group, "split": pair_split})
    hard_candidates = [i for i in original_indices if samples[i]["family"] == duplicate["family"] and samples[i]["true_defect"] == duplicate["true_defect"] and samples[i]["duplicate_group"] != anchor_group and duplicate_group_split[samples[i]["duplicate_group"]] == pair_split]
    hard_index = int(hard_candidates[stable_seed(duplicate["sample_id"]) % len(hard_candidates)])
    pair_rows.append({"first": original_index, "second": hard_index, "duplicate": 0, "pair_type": "semantic hard negative",
                      "anchor_group": anchor_group, "candidate_group": samples[hard_index]["duplicate_group"], "split": pair_split})
    random_candidates = [i for i in original_indices if samples[i]["family"] != duplicate["family"] and duplicate_group_split[samples[i]["duplicate_group"]] == pair_split]
    random_index = int(random_candidates[stable_seed("random", duplicate["sample_id"]) % len(random_candidates)])
    pair_rows.append({"first": original_index, "second": random_index, "duplicate": 0, "pair_type": "easy negative",
                      "anchor_group": anchor_group, "candidate_group": samples[random_index]["duplicate_group"], "split": pair_split})
duplicate_pairs = pd.DataFrame(pair_rows)
duplicate_pairs["embedding_similarity"] = [float(resnet_features["full"][a] @ resnet_features["full"][b]) for a, b in zip(duplicate_pairs["first"], duplicate_pairs["second"])]
duplicate_pairs["dhash_similarity"] = [hash_similarity(samples[a]["image"], samples[b]["image"]) for a, b in zip(duplicate_pairs["first"], duplicate_pairs["second"])]
development_groups = set(duplicate_pairs.query("split == 'development'")[["anchor_group", "candidate_group"]].to_numpy().ravel())
test_groups = set(duplicate_pairs.query("split == 'test'")[["anchor_group", "candidate_group"]].to_numpy().ravel())
split_group_overlap = development_groups.intersection(test_groups)
assert not split_group_overlap, f"duplicate-group leakage: {sorted(split_group_overlap)}"
duplicate_split_contract = duplicate_pairs.groupby("split").agg(pairs=("duplicate", "size"), positive_pairs=("duplicate", "sum"), anchor_groups=("anchor_group", "nunique"), candidate_groups=("candidate_group", "nunique")).reset_index()
duplicate_split_contract["cross_split_group_overlap"] = len(split_group_overlap)
display(duplicate_split_contract)


def choose_threshold(frame: pd.DataFrame, score_column: str) -> float:
    candidates = np.linspace(frame[score_column].min(), frame[score_column].max(), 101)
    scored = [(f1_score(frame.duplicate, frame[score_column] >= threshold, zero_division=0), threshold) for threshold in candidates]
    return float(max(scored, key=lambda item: (item[0], item[1]))[1])


duplicate_evaluation_rows = []
for score_column in ["embedding_similarity", "dhash_similarity"]:
    threshold = choose_threshold(duplicate_pairs.query("split == 'development'"), score_column)
    test = duplicate_pairs.query("split == 'test'")
    prediction = test[score_column] >= threshold
    duplicate_evaluation_rows.append({
        "method": score_column,
        "development_threshold": threshold,
        "test_precision": precision_score(test.duplicate, prediction, zero_division=0),
        "test_recall": recall_score(test.duplicate, prediction, zero_division=0),
        "test_f1": f1_score(test.duplicate, prediction, zero_division=0),
        "test_pairs": len(test),
    })
duplicate_evaluation = pd.DataFrame(duplicate_evaluation_rows)
display(duplicate_evaluation.round(3))
display(duplicate_pairs.groupby(["pair_type", "split"])[["embedding_similarity", "dhash_similarity"]].mean().round(3))


## 9. Hard-negative review queue

High similarity plus disagreeing observed labels is a review trigger—not an automatic relabel. The queue distinguishes duplicate, likely label error, same-source shortcut, hard negative, and taxonomy ambiguity using available metadata.


In [ ]:
def classify_review_reason(query: dict, neighbor: dict) -> str:
    if query["duplicate_group"] == neighbor["duplicate_group"]: return "duplicate"
    if query["true_defect"] == neighbor["true_defect"] and query["observed_defect"] != neighbor["observed_defect"]: return "likely label error / false negative"
    if query["source"] == neighbor["source"] and query["true_defect"] != neighbor["true_defect"]: return "same-source shortcut"
    if query["family"] == neighbor["family"]: return "hard negative"
    return "ambiguous taxonomy / representation mismatch"


review_features = resnet_features["full"]
review_scores = review_features @ review_features.T
np.fill_diagonal(review_scores, -np.inf)
review_pairs = set()
for query_index, query in enumerate(samples):
    candidate_groups = [
        [i for i, neighbor in enumerate(samples) if neighbor["observed_defect"] != query["observed_defect"]],
        [i for i, neighbor in enumerate(samples) if i != query_index and neighbor["duplicate_group"] == query["duplicate_group"]],
        [i for i, neighbor in enumerate(samples) if neighbor["true_defect"] != query["true_defect"] and
         neighbor["family"] == query["family"] and neighbor["source"] != query["source"]],
        [i for i, neighbor in enumerate(samples) if neighbor["observed_defect"] != query["observed_defect"] and
         neighbor["family"] != query["family"] and neighbor["source"] != query["source"]],
        [i for i, neighbor in enumerate(samples) if neighbor["true_defect"] == query["true_defect"] and
         neighbor["observed_defect"] != query["observed_defect"]],
    ]
    for candidates in candidate_groups:
        if candidates:
            neighbor_index = max(candidates, key=lambda i: review_scores[query_index, i])
            review_pairs.add((query_index, neighbor_index))

hard_negative_rows = []
for query_index, neighbor_index in sorted(review_pairs):
    query, neighbor = samples[query_index], samples[neighbor_index]
    hard_negative_rows.append({
        "query_id": query["sample_id"], "neighbor_id": neighbor["sample_id"],
        "similarity": float(review_scores[query_index, neighbor_index]),
        "query_label": query["observed_defect"], "neighbor_label": neighbor["observed_defect"],
        "query_true_label": query["true_defect"], "neighbor_true_label": neighbor["true_defect"],
        "query_source": query["source"], "neighbor_source": neighbor["source"],
        "query_family": query["family"], "neighbor_family": neighbor["family"],
        "review_reason": classify_review_reason(query, neighbor),
    })
hard_negative_review = pd.DataFrame(hard_negative_rows).sort_values("similarity", ascending=False).reset_index(drop=True)
display(hard_negative_review.head(20))
display(hard_negative_review.review_reason.value_counts().rename_axis("reason").to_frame("count"))


## 10. Exact search: NumPy, scikit-learn, and FAISS parity

Exact search is the ground-truth baseline. For normalized vectors, NumPy matrix multiplication, scikit-learn cosine distance, and FAISS `IndexFlatIP` should produce the same top neighbours. This assertion catches normalization, direction, dtype, and ID-order mistakes before approximate search is introduced.

The FAISS calls run in a short-lived local worker process. This keeps its platform-native parallel runtime separate from the long-lived PyTorch/Jupyter process on systems where loading both runtimes into one kernel is unstable. The worker uses the public FAISS Python SDK, local `.npz` inputs, and no service or network call; all worker source remains visible in this notebook.


In [ ]:
exact_gallery = offline_features[gallery_indices].astype(np.float32)
exact_queries = offline_features[query_indices[:20]].astype(np.float32)
K_EXACT = 10

numpy_scores = exact_queries @ exact_gallery.T
numpy_ids = np.argsort(-numpy_scores, axis=1)[:, :K_EXACT]

sklearn_index = NearestNeighbors(n_neighbors=K_EXACT, metric="cosine", algorithm="brute")
sklearn_index.fit(exact_gallery)
_, sklearn_ids = sklearn_index.kneighbors(exact_queries)

FAISS_WORKER_SOURCE = r'''\
import json
import sys
import time
import faiss
import numpy as np

action, input_path, output_path = sys.argv[1:4]
data = np.load(input_path)
if action == "exact":
    gallery = data["gallery"].astype(np.float32)
    queries = data["queries"].astype(np.float32)
    k = int(data["k"])
    index = faiss.IndexFlatIP(gallery.shape[1])
    index.add(gallery)
    scores, ids = index.search(queries, k)
    np.savez(output_path, scores=scores, ids=ids, flat_size=np.array(faiss.serialize_index(index).nbytes))
elif action == "hnsw":
    gallery = data["gallery"].astype(np.float32)
    queries = data["queries"].astype(np.float32)
    ef_values = data["ef_values"].astype(int)
    k = int(data["k"])
    exact = faiss.IndexFlatIP(gallery.shape[1]); exact.add(gallery)
    _, exact_ids = exact.search(queries, k)
    hnsw = faiss.IndexHNSWFlat(gallery.shape[1], 16, faiss.METRIC_INNER_PRODUCT)
    hnsw.hnsw.efConstruction = 80
    hnsw.add(gallery)
    returned, timing_rows = [], []
    for ef in ef_values:
        hnsw.hnsw.efSearch = int(ef)
        hnsw.search(queries[:5], k)
        _, ids = hnsw.search(queries, k)
        timings = []
        for _ in range(3):
            for query in queries:
                started = time.perf_counter_ns()
                hnsw.search(query.reshape(1, -1), k)
                timings.append((time.perf_counter_ns() - started) / 1e6)
        returned.append(ids); timing_rows.append(timings)
    np.savez(
        output_path,
        exact_ids=exact_ids,
        returned=np.stack(returned),
        timings=np.asarray(timing_rows),
        ef_values=ef_values,
        flat_size=np.array(faiss.serialize_index(exact).nbytes),
        hnsw_size=np.array(faiss.serialize_index(hnsw).nbytes),
    )
else:
    raise ValueError(action)
print(json.dumps({"faiss": faiss.__version__, "action": action}))
'''


def run_faiss_worker(action: str, **arrays):
    input_path = ARTIFACT_DIR / f"faiss-{action}-input.npz"
    output_path = ARTIFACT_DIR / f"faiss-{action}-output.npz"
    np.savez(input_path, **arrays)
    completed = subprocess.run(
        [sys.executable, "-c", FAISS_WORKER_SOURCE, action, str(input_path), str(output_path)],
        check=True,
        capture_output=True,
        text=True,
    )
    return np.load(output_path), json.loads(completed.stdout)


faiss_exact_result, faiss_exact_worker = run_faiss_worker(
    "exact", gallery=exact_gallery, queries=exact_queries, k=np.array(K_EXACT)
)
faiss_scores, faiss_ids = faiss_exact_result["scores"], faiss_exact_result["ids"]

assert np.array_equal(numpy_ids, sklearn_ids)
assert np.array_equal(numpy_ids, faiss_ids)
exact_parity = {
    "numpy_vs_sklearn_top10": True,
    "numpy_vs_faiss_top10": True,
    "max_score_error_numpy_faiss": float(np.max(np.abs(np.take_along_axis(numpy_scores, numpy_ids, axis=1) - faiss_scores))),
    "gallery_vectors": len(exact_gallery),
    "dimension": exact_gallery.shape[1],
    "execution_boundary": "isolated local FAISS worker process",
    "worker": faiss_exact_worker,
}
print(exact_parity)


## 11. FAISS HNSW recall–latency sweep

The tiny teaching corpus is too small to motivate ANN, so we create a deterministic scale proxy by perturbing learned vectors. Exact `IndexFlatIP` supplies top-10 ground truth. `IndexHNSWFlat` searches the same normalized float32 vectors while `efSearch` changes graph exploration depth. For each setting, the worker times 300 **individual sequential query calls** (100 queries × 3 repeats) and reports their median and p95. This includes local Python/FAISS call overhead, but it is not a concurrent service tail-latency or real large-corpus capacity claim.

![Exact search scans every vector; approximate search visits a candidate subset and must be evaluated against exact neighbours.](assets/exact-vs-ann.svg)


In [ ]:
rng = np.random.default_rng(SEED)
base_pool = offline_features[original_indices]
selected = rng.integers(0, len(base_pool), size=RUN["ann_vectors"])
ann_gallery = l2_normalize((base_pool[selected] + rng.normal(0, 0.07, (RUN["ann_vectors"], base_pool.shape[1]))).astype(np.float32))
query_base = rng.choice(len(base_pool), size=RUN["ann_queries"], replace=True)
ann_queries = l2_normalize((base_pool[query_base] + rng.normal(0, 0.025, (RUN["ann_queries"], base_pool.shape[1]))).astype(np.float32))

def ann_recall_at_k(approximate: np.ndarray, exact: np.ndarray) -> float:
    return float(np.mean([len(set(a).intersection(e)) / exact.shape[1] for a, e in zip(approximate, exact)]))


ef_values = np.array([4, 8, 16, 32, 64])
ann_worker_result, ann_worker_metadata = run_faiss_worker(
    "hnsw", gallery=ann_gallery, queries=ann_queries, k=np.array(10), ef_values=ef_values
)
exact_top10 = ann_worker_result["exact_ids"]
ann_rows = []
for row_index, ef_search in enumerate(ann_worker_result["ef_values"]):
    timings = ann_worker_result["timings"][row_index]
    returned = ann_worker_result["returned"][row_index]
    ann_rows.append({
        "efSearch": int(ef_search),
        "ANN_Recall@10": ann_recall_at_k(returned, exact_top10),
        "median_individual_query_ms": float(np.median(timings)),
        "p95_individual_query_ms": float(np.percentile(timings, 95)),
        "individual_timing_samples": len(timings),
        "timing_scope": "sequential one-query FAISS calls; not concurrent service latency",
        "queries": len(ann_queries),
        "corpus_vectors": len(ann_gallery),
    })
ann_recall_latency = pd.DataFrame(ann_rows)
display(ann_recall_latency.round(4))

fig, axis = plt.subplots(figsize=(6, 4))
axis.plot(ann_recall_latency["median_individual_query_ms"], ann_recall_latency["ANN_Recall@10"], marker="o")
for _, row in ann_recall_latency.iterrows(): axis.annotate(f"ef={int(row.efSearch)}", (row.median_individual_query_ms, row["ANN_Recall@10"]), fontsize=8)
axis.set(xlabel="median sequential individual-query ms", ylabel="ANN Recall@10", title="HNSW recall–latency trade-off")
axis.grid(alpha=.25); plt.tight_layout(); plt.show()


In [ ]:
raw_vector_bytes = int(ann_gallery.size * ann_gallery.dtype.itemsize)
memory_estimate = {
    "vectors": len(ann_gallery),
    "dimension": ann_gallery.shape[1],
    "raw_float32_vector_bytes": raw_vector_bytes,
    "flat_serialized_bytes": int(ann_worker_result["flat_size"]),
    "hnsw_serialized_bytes": int(ann_worker_result["hnsw_size"]),
    "hnsw_to_raw_ratio": int(ann_worker_result["hnsw_size"]) / raw_vector_bytes,
    "metadata_dataframe_bytes_for_course_corpus": int(metadata.memory_usage(index=True, deep=True).sum()),
    "ten_million_768d_float32_gb": 10_000_000 * 768 * 4 / 1e9,
}
display(pd.Series(memory_estimate, name="measured_or_formula").to_frame())
print("Serialized bytes are measured; allocator/process/replica overhead is not inferred.")


## 12. Metadata filtering: pre-filter versus post-filter

The example asks for the same component family from a different source. Pre-filtering searches only eligible IDs. Post-filtering searches a larger vector index and discards unauthorized/ineligible results; it may return fewer than K unless candidate oversampling is sufficient.


In [ ]:
filter_query_index = next(i for i in original_indices if samples[int(i)]["source"] == "B")
filter_query = samples[filter_query_index]
filter_features = resnet_features["full"]
filter_query_vector = filter_features[filter_query_index]
eligible_gallery = np.array([i for i in original_indices if samples[i]["source"] != filter_query["source"] and samples[i]["family"] == filter_query["family"]])
all_other = np.array([i for i in original_indices if i != filter_query_index])
K_FILTER = 5

pre_started = time.perf_counter_ns()
pre_order = eligible_gallery[np.argsort(-(filter_features[eligible_gallery] @ filter_query_vector))[:K_FILTER]]
pre_ms = (time.perf_counter_ns() - pre_started) / 1e6

shallow_started = time.perf_counter_ns()
shallow_ranked_all = all_other[np.argsort(-(filter_features[all_other] @ filter_query_vector))]
shallow_candidates = shallow_ranked_all[:K_FILTER]
shallow_post_order = np.array([i for i in shallow_candidates if samples[i]["source"] != filter_query["source"] and samples[i]["family"] == filter_query["family"]])[:K_FILTER]
shallow_post_ms = (time.perf_counter_ns() - shallow_started) / 1e6

mitigated_started = time.perf_counter_ns()
mitigated_ranked_all = all_other[np.argsort(-(filter_features[all_other] @ filter_query_vector))]
oversampled = mitigated_ranked_all[:80]
post_order = np.array([i for i in oversampled if samples[i]["source"] != filter_query["source"] and samples[i]["family"] == filter_query["family"]])[:K_FILTER]
mitigated_post_ms = (time.perf_counter_ns() - mitigated_started) / 1e6

filtered_retrieval = pd.DataFrame([
    {"strategy": "pre-filter", "eligible_candidates": len(eligible_gallery), "returned": len(pre_order), "elapsed_ms": pre_ms,
     "top_ids": [samples[i]["sample_id"] for i in pre_order]},
    {"strategy": "post-filter after top-5 (no oversampling)", "eligible_candidates": len(all_other), "returned": len(shallow_post_order), "elapsed_ms": shallow_post_ms,
     "top_ids": [samples[i]["sample_id"] for i in shallow_post_order]},
    {"strategy": "post-filter after top-80 (mitigation)", "eligible_candidates": len(all_other), "returned": len(post_order), "elapsed_ms": mitigated_post_ms,
     "top_ids": [samples[i]["sample_id"] for i in post_order]},
])
display(filtered_retrieval)
shallow_filter_overlap = len(set(pre_order).intersection(shallow_post_order)) / K_FILTER
mitigated_filter_overlap = len(set(pre_order).intersection(post_order)) / K_FILTER
filter_induced_miss = len(shallow_post_order) < K_FILTER or shallow_filter_overlap < 1.0
print({"shallow_top5_overlap": shallow_filter_overlap, "mitigated_top5_overlap": mitigated_filter_overlap,
       "filter_induced_miss": filter_induced_miss})


## 13. Embedding versions, drift, and blue/green migration

ResNet v1 produces 512-dimensional vectors; the domain encoder v2 produces 32-dimensional vectors. Their coordinates are not comparable. Even when dimensions match, new weights or preprocessing define a new space unless compatibility was trained and tested.

Within v2, crop versus masked preprocessing keeps dimension fixed but changes neighbourhoods. We measure corresponding-vector cosine, top-5 neighbour retention, and source/semantic rates rather than treating geometry movement as automatic task degradation.

![Encoder v1 and v2 produce separate spaces; a blue/green migration re-embeds, builds, validates, switches, and later retires the old index.](assets/embedding-versioning.svg)


In [ ]:
encoder_manifests = {
    "encoder_v1": {
        "model": "torchvision ResNet-18",
        "weights": weights.name,
        "dimension": resnet_dimension,
        "region_policy": "full image",
        "normalization": "L2 after encoder",
        "distance": "cosine / inner product",
    },
    "encoder_v2": {
        "model": "TinyMetricEncoder",
        "training_objective": "triplet margin",
        "sampling": "random online then oracle-adjudicated offline hard negatives in the synthetic lab",
        "dimension": offline_features.shape[1],
        "region_policy": "oracle masked region",
        "normalization": "inside model + validation",
        "distance": "cosine / inner product",
    },
}
try:
    _ = resnet_features["full"] @ offline_features.T
    cross_version_status = "unexpectedly comparable"
except ValueError as error:
    cross_version_status = f"blocked: {type(error).__name__} (512D and 32D spaces)"
assert cross_version_status.startswith("blocked")

v2_crop_features = extract_metric_features(offline_model, samples, mode="crop")


def neighbor_retention(first: np.ndarray, second: np.ndarray, indices: np.ndarray, k: int = 5) -> float:
    def neighbors(features):
        scores = features[indices] @ features[indices].T
        np.fill_diagonal(scores, -np.inf)
        return np.argsort(-scores, axis=1)[:, :k]
    first_neighbors, second_neighbors = neighbors(first), neighbors(second)
    return float(np.mean([len(set(a).intersection(b)) / k for a, b in zip(first_neighbors, second_neighbors)]))


corresponding_cosine = np.sum(offline_features[original_indices] * v2_crop_features[original_indices], axis=1)
embedding_drift = {
    "comparison": "encoder_v2 masked preprocessing versus crop preprocessing",
    "mean_corresponding_cosine": float(corresponding_cosine.mean()),
    "p10_corresponding_cosine": float(np.percentile(corresponding_cosine, 10)),
    "top5_neighbor_retention": neighbor_retention(offline_features, v2_crop_features, original_indices),
    "masked_rates": neighbor_rates(offline_features, original_indices),
    "crop_rates": neighbor_rates(v2_crop_features, original_indices),
    "interpretation": "geometry movement is evaluated alongside task/source neighbour behavior",
}
migration_plan = [
    "freeze encoder/index/filter manifests",
    "re-embed governed corpus with encoder_v2",
    "build index_v2 in parallel",
    "validate task metrics, ANN recall, filters, privacy, latency, and drift",
    "shadow queries and inspect failure slices",
    "switch versioned alias with rollback available",
    "monitor, propagate deletion, then retire index_v1",
]
print(json.dumps({"cross_version": cross_version_status, "drift": embedding_drift, "migration": migration_plan}, indent=2))


## 14. Optional official DINOv2 feature adapter

Set `CV_ENABLE_DINOV2=1` only after reviewing the official source, model card, Apache-2.0 license, checkpoint provenance, cache, preprocessing, memory, and network policy. The repository revision is pinned to the same reviewed commit used by Course 04. Pinning source does not pin downloaded checkpoint bytes; an authorized run must record the resolved artifact hash.


In [ ]:
DINOV2_REPO_REVISION = "7764ea0f912e53c92e82eb78a2a1631e92725fc8"
RUN_DINOV2 = os.getenv("CV_ENABLE_DINOV2", "0") == "1"
optional_dinov2_observations = {
    "status": "not_run",
    "foundation_model": True,
    "model": "dinov2_vits14",
    "official_repository": "facebookresearch/dinov2",
    "repository_revision": DINOV2_REPO_REVISION,
    "license": "Apache-2.0 according to the official model card; recheck before use",
    "reason": "optional network-dependent foundation checkpoint is disabled by default",
    "observations": [],
}
if RUN_DINOV2:
    dinov2 = torch.hub.load(
        f"facebookresearch/dinov2:{DINOV2_REPO_REVISION}",
        "dinov2_vits14",
        source="github",
        trust_repo=True,
    ).eval().to(DEVICE)
    dino_transform = transforms.Compose([
        transforms.Resize((224, 224), antialias=True),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])
    with torch.inference_mode():
        dino_batch = torch.stack([dino_transform(region_image(samples[i], "masked")) for i in query_indices[:8]])
        dino_features = F.normalize(dinov2(dino_batch.to(DEVICE)), dim=1).cpu().numpy()
    optional_dinov2_observations.update({"status": "executed", "dimension": int(dino_features.shape[1]), "queries": len(dino_features)})
print(optional_dinov2_observations)


## 15. Automated failure taxonomy and enterprise evidence artifact

The final failure table combines observed neighbour evidence with explicit cross-version and filter events. The JSON keeps locally measured evidence separate from the skipped optional foundation path and unresolved production assumptions.

![Retrieval failures include semantic mismatch, source shortcut, duplicate leakage, rare-class miss, cross-version incompatibility, and filter-induced miss.](assets/retrieval-failure-taxonomy.svg)


In [ ]:
failure_events = []
for _, row in hard_negative_review.iterrows():
    if row.review_reason == "duplicate": failure_type = "duplicate leakage"
    elif row.review_reason == "same-source shortcut": failure_type = "same-source shortcut"
    elif row.review_reason == "hard negative": failure_type = "hard negative"
    elif "label error" in row.review_reason: failure_type = "wrong semantic neighbor / likely label error"
    else: failure_type = "wrong semantic neighbor"
    query = samples[sample_lookup[row.query_id]]
    if query["true_defect"] == "corrosion" and failure_type == "wrong semantic neighbor": failure_type = "rare-class miss"
    failure_events.append({"failure_type": failure_type, "query_id": row.query_id, "neighbor_id": row.neighbor_id,
                           "similarity": row.similarity, "evidence": row.review_reason})
    if row.query_true_label == "corrosion" and row.neighbor_true_label != "corrosion":
        failure_events.append({"failure_type": "rare-class miss", "query_id": row.query_id, "neighbor_id": row.neighbor_id,
                               "similarity": row.similarity, "evidence": "corrosion query retrieved a different true defect"})
failure_events.append({"failure_type": "cross-version incompatibility", "query_id": None, "neighbor_id": None,
                       "similarity": None, "evidence": cross_version_status})
if filter_induced_miss:
    failure_events.append({"failure_type": "filter-induced miss", "query_id": filter_query["sample_id"], "neighbor_id": None,
                           "similarity": None,
                           "evidence": f"shallow post-filter returned={len(shallow_post_order)}; overlap={shallow_filter_overlap:.2f}; top-80 mitigation overlap={mitigated_filter_overlap:.2f}"})
retrieval_failure_taxonomy = pd.DataFrame(failure_events)
failure_summary = retrieval_failure_taxonomy.groupby("failure_type").size().rename("count").reset_index()
display(failure_summary)
display(retrieval_failure_taxonomy.head(15))


def json_ready(value):
    if isinstance(value, pd.DataFrame): return json_ready(value.to_dict(orient="records"))
    if isinstance(value, pd.Series): return json_ready(value.to_dict())
    if isinstance(value, dict): return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray): return json_ready(value.tolist())
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)): return bool(value)
    return value


index_manifest = {
    "exact": {"type": "FAISS IndexFlatIP", "vectors": len(ann_gallery), "dimension": ann_gallery.shape[1], "metric": "inner product on L2-normalized float32"},
    "approximate": {"type": "FAISS IndexHNSWFlat", "M": 16, "efConstruction": 80, "search_parameter": "efSearch", "remove_support_warning": "review update/delete strategy"},
}
evidence = {
    "locally_measured_evidence": {
        "environment": VERSIONS,
        "similarity_contracts": {
            "component": "same component family",
            "defect": "same hidden true defect",
            "identity": "same manufactured identity across sources",
            "duplicate": "same underlying capture / duplicate group",
        },
        "dataset_contract": {"samples": len(samples), "sources": SOURCES, "factory_C_training_updates": False,
                             "near_duplicates": int(metadata.is_near_duplicate.sum()), "controlled_label_errors": int(metadata.label_error.sum())},
        "metric_math": {"normalization_identity_max_error": normalization_identity_error},
        "encoder_versions": encoder_manifests,
        "embedding_dimensions": {"resnet18": resnet_dimension, "tiny_metric": offline_features.shape[1]},
        "distance_metric": "cosine / inner product over L2-normalized vectors",
        "training_objective": {"name": "triplet margin", "margin": 0.35, "target_contract": "defect similarity"},
        "sampling_policy": training_history.groupby("policy").agg(final_loss=("loss", "last"), mean_active_fraction=("active_fraction", "mean")).reset_index(),
        "sampler_learning_signal": sampler_signal,
        "retrieval_metrics": representation_comparison,
        "offline_mining": {"before_after": offline_comparison, "adjudication": "simulated hidden-label oracle; production requires human/domain review",
                           "mined": len(offline_triplets), "approved": len(approved),
                           "false_negative_rate": float(offline_triplets.false_negative.mean())},
        "region_and_source_bias": source_bias,
        "duplicate_detection": {"metrics": duplicate_evaluation, "split_contract": duplicate_split_contract,
                                "split_unit": "duplicate_group / underlying image identity before pair construction",
                                "cross_split_group_overlap": len(split_group_overlap)},
        "hard_negative_review": hard_negative_review.head(50),
        "hard_negative_review_summary": hard_negative_review.review_reason.value_counts().rename_axis("reason").reset_index(name="count"),
        "exact_search_parity": exact_parity,
        "index_manifest": index_manifest,
        "ann_recall_latency": ann_recall_latency,
        "memory_estimate": memory_estimate,
        "filter_policy": {"rule": "source != query source AND same component family", "comparison": filtered_retrieval,
                          "shallow_top5_overlap": shallow_filter_overlap, "mitigated_top5_overlap": mitigated_filter_overlap,
                          "filter_induced_miss": filter_induced_miss},
        "migration_plan": migration_plan,
        "embedding_drift": embedding_drift,
        "failure_taxonomy": retrieval_failure_taxonomy,
        "limitations": [
            "procedural images are not a production manufacturing benchmark",
            "oracle crop and mask rows exclude detector/segmenter error",
            "small CPU timing does not predict a production service SLO",
            "HNSW scale-up uses perturbed copies of a small learned corpus; it validates methodology, not real large-corpus graph, cache, concurrency, or workload behavior",
            "one tiny triplet model does not represent all metric-learning methods",
            "duplicate thresholds require recalibration for every corpus and encoder version",
        ],
    },
    "optional_downloaded_model_observations": {"dinov2": optional_dinov2_observations},
    "unresolved_production_assumptions": {
        "demonstration_contract": DEMONSTRATION_CONTRACT,
        "requires_target_evidence": ["archive scale and churn", "concurrent query distribution", "target hardware latency/memory",
                                     "tenant authorization", "deletion propagation", "backup/restore", "privacy/legal review", "human relevance judgments"],
    },
}

artifact_path = ARTIFACT_DIR / "course-07-retrieval-evidence.json"
artifact_path.write_text(json.dumps(json_ready(evidence), indent=2) + "\n", encoding="utf-8")
for name, frame in {
    "training-history.csv": training_history,
    "representation-comparison.csv": representation_comparison,
    "source-bias.csv": source_bias,
    "duplicate-evaluation.csv": duplicate_evaluation,
    "hard-negative-review.csv": hard_negative_review,
    "ann-recall-latency.csv": ann_recall_latency,
    "retrieval-failures.csv": retrieval_failure_taxonomy,
}.items():
    frame.to_csv(ARTIFACT_DIR / name, index=False)
print(f"saved {artifact_path} ({artifact_path.stat().st_size:,} bytes)")
print("top-level evidence partitions:", list(evidence))


## 16. Production upgrade, exercises, and summary

| Teaching system | Production upgrade |
| --- | --- |
| Procedural archive | licensed, consented, versioned media with identity/source/time/tenant groups |
| Oracle crop/mask | end-to-end predicted regions with perturbation and failure attribution |
| Tiny triplet encoder | controlled frozen/domain/adapted model study with repeated seeds and human relevance judgments |
| One process | versioned embedding service, index build pipeline, authorization, replicas, backup/restore, rollback |
| Small HNSW sweep | target-scale workload, concurrency, warm/cold tails, update/delete tests, capacity and cost |
| Local JSON/CSV | artifact registry, lineage, access control, retention, review decisions, release gates, alerts with owners |

Exercises:

1. Implement supervised contrastive loss under the same `P × K` batches.
2. Add an IVF index and sweep `nprobe` against the same exact top-10.
3. Perturb oracle boxes and measure region-retrieval degradation.
4. Add tenant IDs and assertion-test that unauthorized records never enter candidates.
5. Replace the demonstration contract with evidence from one defined target device and workload.

The durable chain is:

```text
image / object / region → encoder → embedding → similarity geometry
→ metric learning → exact / approximate neighbours → retrieval evaluation
→ versioned production search + monitoring
```

Course 08 adds time: the nearest visual item becomes a candidate identity association, then tracking must manage motion, occlusion, lifecycle, and identity switches.
